# Image embedding category score

In [57]:
import pandas as pd
import numpy as np
import random as rd
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

## 1. Read files

In [58]:
ITEM_PATH = '../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.item'
IMAGE_ORIGINAL_PATH = '../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.image_original'

df_item = pd.read_csv(ITEM_PATH, sep='\t')
df_image = pd.read_csv(IMAGE_ORIGINAL_PATH, sep='\t')

## 2. Preprocess data

In [59]:
bool_features_per_item = np.vstack(
    df_image['ent_emb:float_seq']
    .str.split()
    .apply(lambda xs: [int(float(x) != 0.0) for x in xs])
    .to_list()
)

categories_per_item = (df_item['categories:token_seq']
    .apply(lambda x: eval(f'[{x}]'))
    .tolist()
)

for categories in categories_per_item:
    categories.remove('Sports & Outdoors')

## 3. Fit KNN with static embeddings

In [60]:
knn = NearestNeighbors(n_neighbors=10, metric='cosine')
knn.fit(bool_features_per_item)

,n_neighbors,10
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


## 4. Metric

In [61]:
def score_func(x, ys):
    main_set = set(categories_per_item[x])
    return sum(len(main_set & set(categories_per_item[y])) for y in ys) / len(main_set) / len(ys) if len(main_set) > 0 else 0

## 5. KNN analysis

In [62]:
ids = range(len(categories_per_item))
BATCH_SIZE = 1000
knn_score = 0.0

for L in tqdm(range(0, len(ids), BATCH_SIZE), unit="batch"):
    R = min(L + BATCH_SIZE, len(ids))
    ids_slice = ids[L:R]
    _, nearest_ids = knn.kneighbors([bool_features_per_item[idx] for idx in ids_slice])
    for j, ks in zip(ids_slice, nearest_ids):
        knn_score += score_func(j, ks)

knn_score /= len(ids)
print(f'KNN score: {knn_score}')

100%|██████████| 19/19 [00:15<00:00,  1.19batch/s]

KNN score: 0.42115202467752444


## 6. Random analysis

In [69]:
random_score = 0.0

for main_idx in tqdm(ids):
    selected_ids = rd.sample(ids, k=10)
    random_score += score_func(main_idx, selected_ids)

random_score /= len(ids)
print(f"Random score: {random_score}")

100%|██████████| 18357/18357 [00:00<00:00, 128253.98it/s]

Random score: 0.09998004586008871
